# 환경변수  import

In [67]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os
from langchain_openai.chat_models.base import ChatOpenAI
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.runnables.passthrough import RunnablePassthrough

# RAG
Retrieval Augmented Generation (검색증강생성)

In [4]:
# 사전학습된 모델은 이미 많은 데이터를 통해 학습한 상태이긴 하나..
# 개인 DB 나 회사내 문서 와 같이 'private 한 데이터' 들에는 접근할수 없다
# 그래서 RAG 를 사용한다!

In [5]:
# 1. Retrieval 단계
# private 으로부터 제공된 data 를 사용하거나 탐색함으로써
# language model 의 능력을 더 '확장(augment)'

# 2. Augmented Generation
# Model 로 하여금 '우리가 보낸 문서 data 만'을 가지고 답변하도록 할수도 있다.
# (경우에 따라, 우리 문서가 더 최신 data 일수도 있기 때문이다)
# 이를 통해 Model 이 과거에 학습한 data 를 참조하지 않게도 할수 있다.

In [6]:
# RAG 는 특정 라이브러리나 프레임워크 이름이 아니라
# 위와 같은 작업을 하는 '기법'을 일반적으로 통칭하는 용어

# RAG 를 수행하는 방법은 굉~장히 많고 다양.


In [7]:
# 어떤 방식으로 RAG 를 구현할른지는

# - 우리가 얼마나 많은 문서들을 가지고 있는지
# - 우리가 얼마나 많은 비용으로 운영할지 (어떤 모델, 가용한 token 개수등..)

# 등에 따라 결정될 문제다.

# Retrieval 

![](https://miro.medium.com/v2/resize:fit:1100/format:webp/0*jmVYxqojDFn2yoOr.jpg)

In [8]:
# RAG 의 첫번째 단계인 Retrieval 의 일반적인 과정
# - data source 에서 데이터 load
# - 데이터는 split 하면서 transform
# - transform 한 데이터를 embed.
# - embed 된 데이터를 store 에 저장.
# - 검색(질의) 가 입력되면 store 에서 관련 문서들을 retrieve!

# Data Loaders

In [9]:
# 랭체인에서 제공하는 다양한 document loader 들이 있다
# CSV, File Directory, HTML, JSON, Markdown, PDF 등
# ※그 밖에서도 3rd party loader 들도 있다.

## 예제 파일 준비

In [10]:
# 아래와 같이 파일들을 준비합니다

# 출처는  조지오웰의 소설 '1984' Part1 Chapter1
#  http://www.george-orwell.org/1984/0.html

# 너무 길거나, 너무 짧지 않으면 좋습니다
# 파일이 너무 길면 나중에 임베딩 과정에서 비용지출이 발생.

# 다운로드 링크
# https://www.dropbox.com/scl/fi/ppid6hk7bwqxc0xrv65oc/files.zip?rlkey=df5d411n1fnaht2wlwv0o71wt&st=ddd52gw3&dl=1


In [11]:
llm = ChatOpenAI(temperature=0.1)

base_path = r'D:\MCP2601\dataset'

## TextLoader

In [12]:
from langchain_community.document_loaders.text import TextLoader

In [13]:
# Data Loader 객체 생성
loader = TextLoader(os.path.join(base_path, 'files', 'chapter_one.txt'))

In [14]:
# 읽어오기
loader.load()  # -> List[Document] 리턴
# 실제 내용은 page_content 속성에 담겨 있다.

[Document(metadata={'source': 'D:\\MCP2601\\dataset\\files\\chapter_one.txt'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. Th

## PyPDFLoader

In [15]:
from langchain_community.document_loaders.pdf import PyPDFLoader

In [16]:
loader = PyPDFLoader(os.path.join(base_path, 'files', 'chapter_one.pdf'))

loader.load()

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-01-30T23:19:00+09:00', 'author': 'Yeonchul Sung', 'moddate': '2025-01-30T23:19:00+09:00', 'source': 'D:\\MCP2601\\dataset\\files\\chapter_one.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Part 1, Chapter 1 \n \n \nPart One \n \n \n1 \nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his \nchin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through \nthe glass doors of Victory Mansions, though not quickly enough to prevent a swirl of \ngritty dust from entering along with him. \n \nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured \nposter, too large for indoor display, had been tacked to the wall. It depicted simply an \nenormous face, more than a metre wide: the face of a man of about forty-five, with a \nheavy black moustache and ruggedly handsome 

## UnstructuredFileLoader

In [17]:
# 서로 다른 타입의 문서를 읽어오기 위해 각각의 DataLoader 를 사용하기 보다
# UnstructuredFileLoader 라는 것도 사용해볼수 있다. -> 꽤 다양한 포맷의 파일을 읽어올 수 있다

In [18]:
from langchain_community.document_loaders.unstructured import UnstructuredFileLoader

In [19]:
loader = UnstructuredFileLoader(os.path.join(base_path, 'files', 'chapter_one.pdf'))

loader.load()

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_9492\1588605583.py:1: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-unstructured package and should be used instead. To use it run `pip install -U `langchain-unstructured` and import as `from `langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileLoader(os.path.join(base_path, 'files', 'chapter_one.pdf'))
No languages specified, defaulting to English.


[Document(metadata={'source': 'D:\\MCP2601\\dataset\\files\\chapter_one.pdf'}, page_content="Part 1, Chapter 1\n\nPart One\n\n1\n\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his\n\nchin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through\n\nthe glass doors of Victory Mansions, though not quickly enough to prevent a swirl of\n\ngritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured\n\nposter, too large for indoor display, had been tacked to the wall. It depicted simply an\n\nenormous face, more than a metre wide: the face of a man of about forty-five, with a\n\nheavy black moustache and ruggedly handsome features. Winston made for the stairs. It\n\nwas no use trying the lift. Even at the best of times it was seldom working, and at\n\npresent the electric current was cut off during daylight hours. It was part of the economy\n\ndrive in pr

In [20]:
loader = UnstructuredFileLoader(os.path.join(base_path, 'files', 'chapter_one.txt'))

loader.load()

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


[Document(metadata={'source': 'D:\\MCP2601\\dataset\\files\\chapter_one.txt'}, page_content="Part 1, Chapter 1\n\nPart One\n\n1 It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The f

In [21]:
loader = UnstructuredFileLoader(os.path.join(base_path, 'files', 'chapter_one.docx'))

loader.load()

[Document(metadata={'source': 'D:\\MCP2601\\dataset\\files\\chapter_one.docx'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. T

# Splitter

## data 를 split 해야 하는 이유

In [22]:
# .load() 의 리턴값은 List[Document]
#  그런데 지금은 Document 가 1개만 있다.  <- 하나의 Document 로 담겨 있다.

In [23]:
len(loader.load())

1

In [24]:
# 특정 질문에 답해야 하기 위해서, 필요한 '파일의 일부분' 만들 전달해야 할 수도 있다.

#  그래서 문서를 쪼개두어야(split) 한다

# 가령: "Ministry of peace" 를 찾고자 한다면.
# 해당 키워드가 있는 문서(들)만 모델에 넘겨주면 된다.

# 작은 조각들로 쪼개어 두면 필요한 것들을 찾기가 용이해진다.
#  - prompt 도 짧아질거다 (적은 token 사용, 적은 비용.)

# split 하는 방법은 다양하다.s

In [25]:
"""
TextSplitter 계층도

BaseDocumentTransformer --> TextSplitter --> <name>TextSplitter  # Example: CharacterTextSplitter
                                             RecursiveCharacterTextSplitter -->  <name>TextSplitter

https://python.langchain.com/api_reference/text_splitters/index.html

"""
None

## RecursiveCharacterTextSplitter

In [26]:
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

In [27]:
splitter = RecursiveCharacterTextSplitter()

# RecursiveCharacterTextSplitter 는 파일을 split 해주는데
# 문장의 끝이나, 문단의 끝부분마다 끊어준다.
# 문장 중간을 끊지는 않는다.  최대한 문장 중간에서 split 되지 않도록 하려 한다.
# 문장 중간에 짤림으로 의미있는 문장들을 잃고 싶지 않다.

# ↓ splitter 사용방법은 두가지 가 있다.

### 방법1  load() , split_documents()

In [28]:
docs = loader.load()  # List[Document]

In [29]:
len(docs)

1

In [30]:
documents = splitter.split_documents(docs)  # -> split 된 List[Document]
print(len(documents), '개 Document')

documents[:10]

11 개 Document


[Document(metadata={'source': 'D:\\MCP2601\\dataset\\files\\chapter_one.docx'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. T

In [31]:
print(documents[0].page_content)

Part 1, Chapter 1

Part One


1
It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.

The hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine and had a varicose ulcer above his righ

### 방법2: load_and_split()

In [32]:
documents = loader.load_and_split(text_splitter=splitter)  # -> split 된 List[Document]
print(len(documents), '개 Document')
documents[:3]

11 개 Document


[Document(metadata={'source': 'D:\\MCP2601\\dataset\\files\\chapter_one.docx'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. T

In [33]:
print(documents[0].page_content)

Part 1, Chapter 1

Part One


1
It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.

The hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine and had a varicose ulcer above his righ

### chunk_size=

In [ ]:
# 좀 더 작은 Document 를 만들 필요가 있다.
# 모델의 Context Window 가 크지 않은 경우라든지..
# chunk_size= 값으로 조정해보자

In [34]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,  # 얼마나 큰 덩어리로 split 할지 지정.
                    # chunk_size 의 단위는 splitter 마다 다르다.
                    # CharacterTextSplitter 의 chunk_size 는 '문자 개수'
)

documents = loader.load_and_split(text_splitter=splitter)  # -> split 된 List[Document]
print(len(documents), '개 Document')
documents[:3]

3498 개 Document


[Document(metadata={'source': 'D:\\MCP2601\\dataset\\files\\chapter_one.docx'}, page_content='Part 1, Chapter 1\n\nPart One'),
 Document(metadata={'source': 'D:\\MCP2601\\dataset\\files\\chapter_one.docx'}, page_content='1'),
 Document(metadata={'source': 'D:\\MCP2601\\dataset\\files\\chapter_one.docx'}, page_content='It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors')]

In [ ]:
# ↑ 굉장히 잘게 쪼개졌다.

# ↑ 문제점: 문단의 중간이 잘려버렸다 -> 문장의 의미가 파괴된다.

# 작은 덩어리이면서 문장의 중간을 잘라먹지 않는 방법은?
# chunk_overlap=
#    split 할때 앞 조각의 일부를 가져와서 연결해준다.
#    Document 간의 겹치는 부분 생길수 있다.



### chunk_overlap=

In [35]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=50,
)

documents = loader.load_and_split(text_splitter=splitter) 
print(len(documents), '개 Document')

# 확인: 겹치는 부분을 확인해보자.
for document in documents[10:15]:
    print('💛', document.page_content)    

250 개 Document
💛 move. BIG BROTHER IS WATCHING YOU, the caption beneath it ran.
💛 Inside the flat a fruity voice was reading out a list of figures which had something to do with the production of pig-iron. The voice came from an oblong metal plaque like a dulled mirror which
💛 an oblong metal plaque like a dulled mirror which formed part of the surface of the right-hand wall. Winston turned a switch and the voice sank somewhat, though the words were still distinguishable.
💛 though the words were still distinguishable. The instrument (the telescreen, it was called) could be dimmed, but there was no way of shutting it off completely. He moved over to the window: a
💛 it off completely. He moved over to the window: a smallish, frail figure, the meagreness of his body merely emphasized by the blue overalls which were the uniform of the party. His hair was very


In [ ]:
# ↑ Document 간에 겹치는 부분이 있다.
# 앞 Document 의 뒷부분을 가져다가 다음 Document 의 앞에 넣었다.
# 이렇게 하므로 문장의 (의미적) 구조를 크게 해치지 않도록 split 했다.


## CharacterTextSplitter

In [36]:
from langchain_text_splitters.character import CharacterTextSplitter

In [37]:
# CharacterTextSplitter 도 동작방식은 비슷하다
# separator=  : 특정 문자열 찾은 다음 이를 기준으로 분할한다.


In [38]:
splitter = CharacterTextSplitter(
    separator='\n',  # 줄바꿈 문자 기준으로 Document 쪼갬.
    chunk_size=600,
    chunk_overlap=100,
)

documents = loader.load_and_split(text_splitter=splitter) 
print(len(documents), '개 Document')

for document in documents[0:5]:
    print('💛', document.page_content)   

Created a chunk of size 963, which is longer than the specified 600
Created a chunk of size 774, which is longer than the specified 600
Created a chunk of size 954, which is longer than the specified 600
Created a chunk of size 922, which is longer than the specified 600
Created a chunk of size 881, which is longer than the specified 600
Created a chunk of size 821, which is longer than the specified 600
Created a chunk of size 700, which is longer than the specified 600
Created a chunk of size 745, which is longer than the specified 600
Created a chunk of size 735, which is longer than the specified 600
Created a chunk of size 671, which is longer than the specified 600
Created a chunk of size 991, which is longer than the specified 600
Created a chunk of size 990, which is longer than the specified 600
Created a chunk of size 1289, which is longer than the specified 600
Created a chunk of size 1605, which is longer than the specified 600
Created a chunk of size 1900, which is longer 

46 개 Document
💛 Part 1, Chapter 1
Part One
1
It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.
💛 The hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine and had a varicose ulcer 

### length_function=

In [39]:
# splitter 에 length 를 계산하는 함수를 제공해줄수 있다.
#  length_function=   
#    기본적으론 파이썬의 len() 을 사용한다 (디폴트) 

In [ ]:
splitter = CharacterTextSplitter(
    separator='\n',
    chunk_size=600,
    chunk_overlap=100,
    length_function=len,   # <-- chunk 개수 카운트 함수
)

documents = loader.load_and_split(text_splitter=splitter) 
print(len(documents), '개 Document')

for document in documents[0:5]:
    print('💛', document.page_content)  

In [ ]:
# 디폴트로 len() 함수가 동작함. CharacterTextSplitter 에선 '글자의 개수'를 chunk 카운트 함.
# 그러나 LLM 에서 말하는 token 은 문자(letter) 와는 다르다.
# 어떤 경우에는 문자 두개, 혹은 세개...  가 한개의 token 으로 카운트 된다.


# TikToken

In [ ]:
# OpenAI 에서의 token 예시
# https://platform.openai.com/tokenizer
# ↓ model 의 관점에서, 몇개의 token 을 사용하는지 확인해 볼수 있다.

In [ ]:
# OpenAI 모델의 tokenizer 를 우리의 splitter 에 사용 가능!

## from_tiktoken_encoder()

In [40]:
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator='\n',
    chunk_size=600,
    chunk_overlap=100,
)

# 이제 '모델'이 텍스트를 세는 방법과 'splitter'가 텍스트를 세는 방법이 일치 하게 되었다.
# model 에는 입력 limit 이 있기 때문에  (context window), 원하는 텍스트들을 모두 한번에 입력할 수는 없다.
# 그래서 우리 텍스트를 길이 계산할때 model 과 같은 방법으로 계산하는게 더 좋다.


# Vectors

## Embedding 과 Vector

In [ ]:
# Embedding 은 사람이 읽는 텍스트를 컴퓨터가 이해(연산)할 수 있는 숫자들(벡터)로 변환하는 작업이다.
# 우리가 만든 Document 마다 각각의 벡터를 만들어 주게 될겁니다.
# OpenAI 는 크기가 최소 1000차원 이상!의 벡터를 제공해준다.

![](https://miro.medium.com/v2/resize:fit:2000/1*SYiW1MUZul1NvL1kc1RxwQ.png)

In [41]:
"""
3개의 차원을 정의해보자

첫번째 차원을 Masculinity (남성성)
두번째 차원을 Femininity (여성성)
세번째 차원을 Royalty (왕족스러움)

이제 특정 단어에 대한 차원 값(점수)를 줘보자

        Masculinity | Femininity  | Royalty
king   | 0.9        | 0.1         | 1.0
queen  | 0.1        | 0.9         | 1.0
man    | 0.9        | 0.1         | 0.0
woman  | 0.1        | 0.9         | 0.0

이렇게 3차원 벡터에 점수를 매겨 보았다.

단어를 이렇게 벡터로 점수를 매기면 '연산'을 할수 있게 된다.
               Masculinity | Femininity  | Royalty
king - man  =>  0.0        | 0.0         | 1.0  ===> 거의 'royal'

royal + woman => 0.1       | 0.9         | 1.0  ===> 거의 'queen'

★텍스트를 벡터화 하니까 '의미에 대한 연산'이 가능해진다!

"""
None

## word2vec 예시

https://turbomaze.github.io/word2vecjson/

## OpenAIEmbeddings

In [42]:
from langchain_openai.embeddings.base import OpenAIEmbeddings

In [43]:
embedder = OpenAIEmbeddings()   # OpenAI 의 Embedding model 객체

In [44]:
embedder.model

'text-embedding-ada-002'

In [ ]:
# text-embedding-ada-002'   
#  https://platform.openai.com/docs/models/text-embedding-ada-002
#  1M token 당 $0.1

# 임베딩 과정은 모델 호출 발생 -> 비용발생!

In [ ]:
# OpenAIEmbeddings 를 통해
#  embed_documents()  <- 문서를 embed 하는것 뿐만 아니라
#  embed_query()      <- query 도 embed 하는 것이 가능하다.


In [45]:
vector = embedder.embed_query("Hi")  # 모델 호출이 발생, "Hi" 에 대한 벡터리턴 -> List[Float]
print(len(vector))  # 1536 개 차원

1536


In [47]:
print(vector)

[-0.0363546647131443, -0.007160174660384655, -0.03377465531229973, -0.02864069864153862, -0.02679038979113102, 0.03458253666758537, -0.0124635249376297, -0.007837752811610699, 0.0019187183352187276, -0.002667963271960616, 0.02471856400370598, -0.0024643640499562025, -0.005788730923086405, -0.0029920930974185467, 0.00666176388040185, -0.0030214113648980856, 0.03385283797979355, -0.0015408382751047611, 0.021057037636637688, -0.00903654471039772, -0.02173461578786373, 0.010365639813244343, 0.006283883936703205, 0.007081992458552122, -0.012261554598808289, 0.0008380140643566847, 0.005876685492694378, -0.009877001866698265, -0.003068646416068077, -0.02475765533745289, 0.01081518642604351, -0.013786105439066887, -0.024470988661050797, -0.014111863449215889, 0.002454591216519475, -0.018998242914676666, 0.0005786287947557867, -0.011336400173604488, 0.01813824102282524, -0.00996169913560152, 0.013147618621587753, -0.011329885572195053, -0.009147302247583866, -0.009701091796159744, -0.0264516007

In [ ]:
# 단어 / 문장마다 1536 개의 차원으로 된 벡터로 임베딩 된다.

In [48]:
vectors = embedder.embed_documents([
    "hi",
    "how are you",
    "good to meet you",    
])

In [49]:
len(vectors)

3

In [51]:
for vector in vectors:  # 각각은 동일한 크기의 차원으로 임베딩되었다.
    print(len(vector))

1536
1536
1536


In [ ]:
# 코드를 실행할때마다 '매번' 문서 embedding 을 반복해서 수행하는건 매우 비효율적이다
#  => 시간 소요 + 또한 비용 지출

# 대신! 그 embeded 된 결과들을 '저장'해 줄겁니다.
# LangChain 은 embedding 한것들을 캐싱하는 기능을 제공해준다

# '동일 Document'는 가급적 한번만 embedding 해주는게 좋다.


# Vector Store

In [ ]:
# 일단 벡터를 만들고 나서, 그것들을 캐시해주고, vector store 에 넣어주면,
# 우리가 '검색'을 할수 있다.
# 그리하여, '관련있는 문서'들만 찾아낼수 있게 되는 거다

# 랭체인은 다양한 vector store 를 제공한다,  어떤거는 cloud 형태이고, 어떤건 유료이기도 하다.

# 우리는 예제에서 무료로 사용할수 있고 로컬로 저장되는 Chroma 라는 것을 사용해볼겁니다
# 나중에 FAISS 라는 메모리기반 vector store 도 사용해볼거구..
# 클라우드기반의 vector store 인 pinecone 도 사용해 보자.

## Chroma

In [52]:
from langchain_community.vectorstores.chroma import Chroma

In [ ]:
# Chroma 에 전달해야 하는 것
#  1. 'split 된 Document들'
#  2. Embedding model

In [53]:
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator='\n',
    chunk_size=600,
    chunk_overlap=100,
)

docs = loader.load_and_split(text_splitter=splitter)  # split 된 Document 준비

embeddings = OpenAIEmbeddings()  # Embedding model 준비

In [54]:

vectorstore = Chroma.from_documents(docs, embeddings)
# 호출하면 모델 호출 + 비용지출 발생
# Document 의 크기가 클수록 비용도 비례


In [57]:
# 위 생성된 vectorstore 를 사용하여 '유사도' 검색 가능.
# similarity_search 는 기본적으로 4개의 Document 리턴 (k=4)
results = vectorstore.similarity_search("where does winston live?", k=5) # -> List[Document] 리턴
print(len(results), '개 Document 검색됨')
results

5 개 Document 검색됨


[Document(metadata={'source': 'D:\\MCP2601\\dataset\\files\\chapter_one.docx'}, page_content='Part 1, Chapter 1\nPart One\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat 

In [58]:
results[0].page_content

'Part 1, Chapter 1\nPart One\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine and had a varicose ulcer above his rig

In [59]:
results[1].page_content

"The Ministry of Love was the really frightening one. There were no windows in it at all. Winston had never been inside the Ministry of Love, nor within half a kilometre of it. It was a place impossible to enter except on official business, and then only by penetrating through a maze of barbed-wire entanglements, steel doors, and hidden machine-gun nests. Even the streets leading up to its outer barriers were roamed by gorilla-faced guards in black uniforms, armed with jointed truncheons.\nWinston turned round abruptly. He had set his features into the expression of quiet optimism which it was advisable to wear when facing the telescreen. He crossed the room into the tiny kitchen. By leaving the Ministry at this time of day he had sacrificed his lunch in the canteen, and he was aware that there was no food in the kitchen except a hunk of dark-coloured bread which had got to be saved for tomorrow's breakfast. He took down from the shelf a bottle of colourless liquid with a plain white l

In [ ]:
# 위 검색된 Document 들이  LLM 에 전달될것이고
# LLM 은 이 Document 기반의 답변을 할수 있게 될것이다 ! --> RAG

## embedding cache

In [ ]:
# 다시 실행하면 임베딩 결과가 사라진다. 재실행하면 재호출 발생
# -> cache 필요!

In [60]:
from langchain_classic.embeddings import CacheBackedEmbeddings

In [61]:
from langchain_classic.storage import LocalFileStore

In [62]:
base_path

'D:\\MCP2601\\dataset'

In [63]:
# 캐시경로 지정.  여기에 embedding 결과를 저장할 것이다.
cache_dir = LocalFileStore(os.path.join(base_path, '.cache'))

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings,   # embedding model
    cache_dir,  # embedding 결과 저장할 장소
)

# 아까는 두번째 매개변수로 embedding model 을 전달햇지만,
#  이번에는 cached_embeddings 을 전달해주면 된다.
vectorstore = Chroma.from_documents(docs, cached_embeddings)


# ↑ 이렇게 하면,  
#  최초에 Chroma.from_documents() 를 호출할때는 
#     OpenAIEmbeddings 을 사용하여 임베딩. 결과는 cache 함
#  다음에 Chroma.from_documents() 를 호출할때는
#      OpenAIEmbeddings 대신에 미리 cache 되어 있듣 embeddings 를 전달할거다.

# 위코드를 실행하여 우리가 또 파일 embedding 작업을 할때는,
# 1.첫번째로!
#   캐시에 embeddings 가 이미 존재하는지 확인할거다.
# 2.만약 없다면!
#    vector store(Chroma.from_documents) 를 호출할 때
#   문서들(docs) 과 함께 OpenAIEmbeddings 를 사용할거다.


D:\MCP2601\AgentWork\.venv\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [64]:
results = vectorstore.similarity_search("where does winston live")
results[0]

Document(metadata={'source': 'D:\\MCP2601\\dataset\\files\\chapter_one.docx'}, page_content="The Ministry of Love was the really frightening one. There were no windows in it at all. Winston had never been inside the Ministry of Love, nor within half a kilometre of it. It was a place impossible to enter except on official business, and then only by penetrating through a maze of barbed-wire entanglements, steel doors, and hidden machine-gun nests. Even the streets leading up to its outer barriers were roamed by gorilla-faced guards in black uniforms, armed with jointed truncheons.\nWinston turned round abruptly. He had set his features into the expression of quiet optimism which it was advisable to wear when facing the telescreen. He crossed the room into the tiny kitchen. By leaving the Ministry at this time of day he had sacrificed his lunch in the canteen, and he was aware that there was no food in the kitchen except a hunk of dark-coloured bread which had got to be saved for tomorrow

In [66]:
# 생성된 벡터 파일 하나만 보자
import glob
for cached_file in glob.glob(os.path.join(base_path, '.cache', '*')):
    with open(cached_file, 'r') as f:
        print(f.read())
        break

[-0.02385471947491169, -0.009730037301778793, -0.00043383671436458826, -0.018008632585406303, -0.02592436783015728, 0.0008466745493933558, -0.006427334621548653, -0.01970198191702366, -0.020306749269366264, -0.038194429129362106, -0.003205267945304513, 0.04047910496592522, -0.0032875833567231894, -0.014245634898543358, 0.012417892925441265, 0.01773984730243683, 0.04596232995390892, 0.02734893187880516, 0.030534040182828903, -0.0359366312623024, -0.01421875599771738, 0.0009004316525533795, -0.014380027540028095, 0.006239185109734535, -0.03201236203312874, -0.008701932616531849, 0.01600618101656437, -0.022927409037947655, 0.002115006325766444, -0.007693986874073744, 0.0035076516214758158, -0.022940848022699356, 0.0017605454195290804, 0.006494531407952309, -0.03975338488817215, -0.026421621441841125, -0.007855257950723171, -0.018062390387058258, 0.006847312208265066, -0.009515008889138699, 0.013896213844418526, -0.0013153693871572614, -0.0044920789077878, 0.0008340752101503313, -0.0306415

# Langsmith

https://www.langchain.com/langsmith

![](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRejzbKjev2a8d-EKtUU06p84fh5NX_S7dDLA&s)


In [ ]:
# LangSmith 는 (LLM) 기반 애플리케이션의
# 개발, 디버깅, 테스트, 평가, 모니터링, 배포를 지원하는 통합 플랫폼 입니다
# https://www.langchain.com/langsmith

# LangSmith를 사용하면 우리의 체인이 무엇을 하고 있는지 시각적으로 볼수 있다.

# ★ 우선 위 사이트에 회원가입하고 API Key 받아옵니다 ★

In [ ]:
"""
↓ .env 파일에 환경변수 입력 (추가)

OPENAI_API_KEY=xxxx
LANGCHAIN_TRACING_V2=true
LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"
LANGCHAIN_API_KEY=xxxx

※환경변수 변경되면 다시 load 해주자.
"""
None

In [68]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.runnables.passthrough import RunnablePassthrough
from langchain_core.prompts.chat import MessagesPlaceholder
from langchain_core.messages.human import HumanMessage
from langchain_core.messages.ai import AIMessage

llm = ChatOpenAI(temperature=0.1)

chat_history = []

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI talking to a human"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)

def load_memory(_):
    # return {'history': chat_history}
    return chat_history

chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm
# chain = RunnablePassthrough.assign(history=chat_history) | prompt | llm


def invoke_chain(question, history):
    result = chain.invoke({"question": question})  # ★ 체인 실행!
    print(result)
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=result.content))
    
    

invoke_chain("My name is John", chat_history)
invoke_chain("What is my name?", chat_history)

content='Hello John! How can I assist you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 24, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DZbcvtnlJLGLPUq5c6t85565sS0Ha', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019dd403-7048-79f1-80fa-ff90855b8621-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 24, 'output_tokens': 10, 'total_tokens': 34, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
content='Your name is John.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 5, 